# Training and evaluating DistilBERT on quiz level classification with shared scoring

In [17]:
import json
import pandas as pd
import joblib
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, set_seed
import time
from datasets import Dataset
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from safetensors.torch import load_file
from collections import Counter

colab = False

if colab == True:
    from google.colab import drive
    drive.mount('/content/drive')
    data_path = Path("/content/drive/MyDrive/Colab Notebooks/3H1L/data")
    models_path = Path("/content/drive/MyDrive/Colab Notebooks/3H1L/models/DistilBERT_quiz_not_indexed_runs")
else:
    data_path = "../data"
    models_path = "../models/DistilBERT_quiz_not_indexed_runs"

with open(f"{data_path}/train_nn_set.json", "r") as f:
    train_nn_set = json.load(f)

with open(f"{data_path}/validate_set.json", "r") as f:
    validate_set = json.load(f)

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

### Transform data into appropriate format

ml_dataset.json is already the right high level structure, it needs individual headlines to be tokenised

In [2]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

padding = "max_length"
truncation = True
max_length = 50

def tokenize_headline(headline):
    return tokenizer(
        headline,
        padding=padding,
        truncation=truncation,
        max_length=max_length
    )

def tokenize_headlines(target_set):

    modified_set = []

    for item in target_set:
        headlines = [tokenize_headline(headline) for headline in item["headlines"]]
        entry = {
            "date": item["date"],
            "headlines": headlines,
            "answer": item["answer"]
        }
        modified_set.append(entry)

    return modified_set

train_set_DB_shared_ready = tokenize_headlines(train_nn_set)
validate_set_DB_shared_ready = tokenize_headlines(validate_set)
test_set_DB_shared_ready = tokenize_headlines(test_set)

Create Hugging Face Datasets

In [ ]:
train_dataset = Dataset.from_list(train_set_DB_shared_ready)
validate_dataset = Dataset.from_list(validate_set_DB_shared_ready)
test_dataset = Dataset.from_list(test_set_DB_shared_ready)

In [4]:
n_train = len(train_dataset)
n_validate = len(validate_dataset)
n_test = len(test_dataset)

Collate

In [5]:
def collate_quizzes(features):
    input_ids = torch.tensor([
        [headline["input_ids"] for headline in quiz["headlines"]]
        for quiz in features
    ])

    attention_mask = torch.tensor([
        [headline["attention_mask"] for headline in quiz["headlines"]]
        for quiz in features
    ])

    answers = torch.tensor([
        quiz["answer"] for quiz in features
    ])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "answer": answers
    }

### DistilBERT

In [6]:
class SharedScorer(nn.Module):

    def __init__(self, model_name):
        super().__init__()

        self.scorer = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=1
        )

    def forward(self, input_ids, attention_mask, answer=None):

        batch_size, n_headlines, seq_length = input_ids.shape

        input_ids = input_ids.view(batch_size * n_headlines, seq_length)
        attention_mask = attention_mask.view(batch_size * n_headlines, seq_length)

        outputs = self.scorer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        scores = outputs.logits.view(batch_size, n_headlines)

        loss = None

        if answer is not None:
            loss = nn.CrossEntropyLoss()(scores, answer)

        return {
            "loss": loss,
            "logits": scores
        }

In [13]:
features = [train_dataset[i] for i in range(8)]

batch = collate_quizzes(features)

print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("answer:", batch["answer"].shape)
print("answers:", batch["answer"])

input_ids: torch.Size([8, 4, 50])
attention_mask: torch.Size([8, 4, 50])
answer: torch.Size([8])
answers: tensor([3, 1, 1, 0, 1, 1, 0, 0])


In [14]:
model = SharedScorer(model_name)

output = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    answer=batch["answer"]
)

print("logits:", output["logits"].shape)
print("loss:", output["loss"])

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 553.54it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


logits: torch.Size([8, 4])
loss: tensor(1.3812, grad_fn=<NllLossBackward0>)


Model hyperparameters and training (only run in Colab)

In [ ]:
# Set the seed
seed = 42
set_seed(seed)

# Load model
model = SharedScorer(model_name)
num_model_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Define metadata saving functions
def create_run_dir(base=models_path):
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_dir = Path(base) / timestamp
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir

# Create run directories
run_dir = create_run_dir()

# Define training args
output_dir = str(run_dir / "checkpoints")
num_train_epochs=20
per_device_train_batch_size=8
per_device_eval_batch_size=67
eval_strategy="epoch"
# eval_steps=1
save_strategy="epoch"
# save_steps=1
disable_tqdm=True
logging_steps=1
report_to="none"
learning_rate=5e-5 # Default is 5e-5
optim="adamw_torch" # Default is "adamw_torch"
weight_decay=0 # Default is 0
warmup_steps=0 # Default is 0
lr_scheduler_type="linear" # Default is "linear"
load_best_model_at_end=True
metric_for_best_model="eval_loss"
greater_is_better=False
early_stopping_patience=5
label_names=["answer"]
remove_unused_columns=False

# Assign training args
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    eval_strategy=eval_strategy,
    # eval_steps=eval_steps,
    save_strategy=save_strategy,
    disable_tqdm=disable_tqdm,
    logging_steps=logging_steps,
    report_to=report_to,
    learning_rate=learning_rate,
    optim=optim,
    weight_decay=weight_decay,
    warmup_steps=warmup_steps,
    lr_scheduler_type=lr_scheduler_type,
    load_best_model_at_end=load_best_model_at_end,
    metric_for_best_model=metric_for_best_model,
    greater_is_better=greater_is_better,
    label_names=label_names,
    remove_unused_columns=remove_unused_columns
)

# Early stopping option
callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)] # If "None", comment line in Trainer

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validate_dataset,
    data_collator=collate_quizzes,
    callbacks=callbacks
)

# Train
output = trainer.train()

# Best validation loss and the step when it happened
best_step = trainer.state.best_step if hasattr(trainer.state, "best_step") else None
best_metric = trainer.state.best_metric if hasattr(trainer.state, "best_metric") else None

# Tiered run metadata
run_metadata = {
    "Model Information": {
        "Model name": model_name,
        "Number of model parameters": num_model_params,
        "Number of trainable parameters": trainable_params,
    },
    "Dataset Information": {
        "Training set size": n_train,
        "Validation set size": n_validate,
        "Test set size": n_test,
    },
    "Tokenizer Settings": {
        "Truncation setting": truncation,
        "Padding setting": padding,
        "Max length": max_length,
    },
    "Training Hyperparameters": {
        "Number of training epochs": num_train_epochs,
        "Per-device training batch size": per_device_train_batch_size,
        "Per-device eval batch size": per_device_eval_batch_size,
        # "Number of evaluation steps": eval_steps,
        "Output directory": output_dir,
        "Save strategy": save_strategy,
        # "Save steps": save_steps,
        "Disable tqdm": disable_tqdm,
        "Logging steps": logging_steps,
        "Report to": report_to,
        "Learning rate": learning_rate,
        "Optimiser": optim,
        "Weight decay": weight_decay,
        "Warmup steps": warmup_steps,
        "Learning rate scheduler type": lr_scheduler_type,
        "Callbacks": [type(cb).__name__ for cb in callbacks],
        "Seed": seed,
        "Load best model at end": load_best_model_at_end,
        "Metric for best model": metric_for_best_model,
        "Greater is better": greater_is_better,
        "Early stopping patience": early_stopping_patience,
        "Label names": label_names,
        "Remove unused columns": remove_unused_columns
    },
    "Training Results": {
        "Training time and speed": output.metrics,
        "Best model checkpoint": trainer.state.best_model_checkpoint,
        "Best validation loss": trainer.state.best_metric,
    }
}

def convert_numpy(obj):
    if isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    elif isinstance(obj, (np.floating, float)):
        return round(float(obj), 6)
    else:
        return obj

with open(f"{run_dir}/run_metadata.json", "w") as f:
    json.dump(convert_numpy(run_metadata), f, indent=4)

print(f"Run metadata saved to {run_dir}/run_metadata.json")

# Extract log history
log_history = trainer.state.log_history

# Training loss
train_points = [x for x in log_history if "loss" in x]
train_steps = [p["step"] for p in train_points]
train_losses = [p["loss"] for p in train_points]

# Validation loss
val_points = [x for x in log_history if "eval_loss" in x]
val_steps = [p["step"] for p in val_points]
val_losses = [p["eval_loss"] for p in val_points]

# Plot
plt.figure(figsize=(8,5))
plt.plot(train_steps, train_losses, label="Training Loss")
plt.plot(val_steps, val_losses, label="Validation Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(run_dir / "loss_curve.png")
plt.close()

Logging and storing (only run in Colab)

In [ ]:
log_history = trainer.state.log_history

clean_log = []
last_train_loss = None
last_step = None

# Record values at each logging step
for entry in log_history:
    step = entry.get("step")
    if "loss" in entry:
        last_train_loss = entry["loss"]
    if "eval_loss" in entry:
        # Skip if this step is already appended
        if step == last_step:
            continue
        clean_entry = {
            "step": step,
            "train_loss": last_train_loss,
            "eval_loss": entry.get("eval_loss"),
            "epoch": entry.get("epoch")
        }
        clean_log.append(clean_entry)
        last_step = step

# Save JSON
with open(run_dir / "log_history.json", "w") as f:
    json.dump(clean_log, f, indent=2)

# Save CSV
df = pd.DataFrame(clean_log)
df.to_csv(run_dir / "log_history.csv", index=False)

# Save best model
best_model_dir = run_dir / "best_model"
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)

Load model for evaluation

In [10]:
model_path = Path(f"{models_path}/20260923-143542/best_model")

model = SharedScorer(model_name)

state_dict = load_file(model_path / "model.safetensors")

model.load_state_dict(state_dict)
model.eval()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1250.60it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SharedScorer(
  (scorer): DistilBertForSequenceClassification(
    (distilbert): DistilBertModel(
      (embeddings): Embeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (transformer): Transformer(
        (layer): ModuleList(
          (0-5): 6 x TransformerBlock(
            (attention): DistilBertSelfAttention(
              (q_lin): Linear(in_features=768, out_features=768, bias=True)
              (k_lin): Linear(in_features=768, out_features=768, bias=True)
              (v_lin): Linear(in_features=768, out_features=768, bias=True)
              (out_lin): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bi

In [ ]:
test_features = [test_dataset[i] for i in range(len(test_dataset))]
test_batch = collate_quizzes(test_features)

with torch.no_grad():
    outputs = model(
        input_ids=test_batch["input_ids"],
        attention_mask=test_batch["attention_mask"]
    )

logits = outputs["logits"]
predictions = torch.argmax(logits, dim=1).numpy()
answers = np.array(test_batch["answer"])

accuracy_DistilBERT_quiz_shared = np.mean(predictions == answers)
print(f"Quiz-level accuracy with shared scoring: {accuracy_DistilBERT_quiz_shared:.2%}")

Quiz-level accuracy: 49.25%


C:\Users\popro\AppData\Local\Temp\ipykernel_12504\436719225.py:12: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  answers = np.array(test_batch["answer"])


Checking the logit mean values

In [16]:
logits.mean(dim=0)

tensor([-0.4336, -0.5996, -0.5053, -0.2952])

Training accuracy, it's very high so this indicates overfitting

In [22]:
train_features = [train_dataset[i] for i in range(len(train_dataset))]
train_batch = collate_quizzes(train_features)

with torch.no_grad():
    train_outputs = model(
        input_ids=train_batch["input_ids"],
        attention_mask=train_batch["attention_mask"]
    )

train_logits = train_outputs["logits"]
train_predictions = torch.argmax(train_logits, dim=1).numpy()
train_answers = np.array(train_batch["answer"])

train_accuracy_DistilBERT_quiz_shared = np.mean(train_predictions == train_answers)
print(f"Quiz-level accuracy with shared scoring for training set: {train_accuracy_DistilBERT_quiz_shared:.2%}")

Quiz-level accuracy with shared scoring for training set: 81.77%


C:\Users\popro\AppData\Local\Temp\ipykernel_12504\2475370277.py:12: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  train_answers = np.array(train_batch["answer"])


Checking validation accuracy

In [23]:
validate_features = [validate_dataset[i] for i in range(len(validate_dataset))]
validate_batch = collate_quizzes(validate_features)

with torch.no_grad():
    validate_outputs = model(
        input_ids=validate_batch["input_ids"],
        attention_mask=validate_batch["attention_mask"]
    )

validate_logits = validate_outputs["logits"]
validate_predictions = torch.argmax(validate_logits, dim=1).numpy()
validate_answers = np.array(validate_batch["answer"])

validate_accuracy_DistilBERT_quiz_shared = np.mean(validate_predictions == validate_answers)
print(f"Quiz-level accuracy with shared scoring for validation set: {validate_accuracy_DistilBERT_quiz_shared:.2%}")

Quiz-level accuracy with shared scoring for validation set: 50.75%


C:\Users\popro\AppData\Local\Temp\ipykernel_12504\187938630.py:12: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  validate_answers = np.array(validate_batch["answer"])
